# Halloween Library｜六句正式聲線試音

**目前只到dry-run，請把六句清單回傳家長確認，尚未生成。** 不生成Library其餘24段、不生成Park、不重錄Bakery。Rosie固定qwen_rosie_r2與Bakery核准主參考，Princess固定af_bella。

1. 新Colab執行階段選免費T4 GPU，執行安裝格；若已用同一個Bakery環境成功安裝，可跳過安裝。
2. 上傳私人library-trial6-inputs.zip，已含同一份核准Rosie參考音，不需重新選角。輸入包不要公開分享。
3. 執行dry-run格，查看六句並回傳確認。**先停在這裡**。
4. 取得家長對清單的確認後，才手動修改生成格的RUN_GENERATION及確認碼；預設執行整本不會推論。
5. 之後生成成功可在本頁逐句聽、記錄接受或需重錄，下載六音檔與獨立試聽HTML。

免費GPU不保證可取得，無資源時停止，不改付費API。情緒/童真是人工驗收目標，不宣稱模型保證指定年齡或精準情緒；不使用pitch或速度處理。

## 1｜安裝（同一個已成功的Bakery執行階段可跳過）

In [ ]:
import sys, subprocess, os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
subprocess.check_call(['apt-get','-qq','update'])
subprocess.check_call(['apt-get','-qq','install','-y','espeak-ng','sox','libsndfile1'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','qwen-tts==0.1.1','transformers==4.57.3','accelerate==1.12.0','kokoro==0.9.4','soundfile','ipywidgets>=8,<9','huggingface-hub>=0.34,<1'])
print('套件安裝完成。若Colab提示套件已載入，請重新啟動執行階段，再從上傳格繼續。')

## 2｜上傳六句輸入包（已包含核准Rosie參考音）

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, json, uuid, importlib.util, hashlib
LIBRARY_WORK=Path('/content/rosie_library_trial6_inputs_'+uuid.uuid4().hex[:8])
print('請上傳 library-trial6-inputs.zip；不是原Bakery輸出包。')
uploaded=files.upload()
if len(uploaded)!=1:raise ValueError('請只上傳一個Library六句輸入ZIP')
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    members=[i for i in archive.infolist() if not i.is_dir()]
    if sum(i.file_size for i in members)>50_000_000 or len({i.filename for i in members})!=len(members):raise ValueError('輸入ZIP不合法')
    paths=[(i,(LIBRARY_WORK/i.filename).resolve()) for i in members]
    if any(not p.is_relative_to(LIBRARY_WORK.resolve()) for _,p in paths):raise ValueError('不安全的ZIP路徑')
    for info,target in paths:
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(archive.read(info))
spec=importlib.util.spec_from_file_location('library_trial6_runner',LIBRARY_WORK/'library_runner.py')
library=importlib.util.module_from_spec(spec);spec.loader.exec_module(library)
library_manifest,library_jobs=library.plan(LIBRARY_WORK)
LIBRARY_OUTPUT=Path('/content')/('library_trial6_output_'+uuid.uuid4().hex[:8])
print('核准Rosie參考音SHA256及逐字稿已驗證。輸出將另存：',LIBRARY_OUTPUT)

## 3｜六句dry-run清單：執行後先停，回傳確認

In [ ]:
dry_run=library.generate(LIBRARY_WORK,LIBRARY_OUTPUT,run=False)
from IPython.display import display, Markdown
import html
rows=['|角色|聲線|固定ID|英文原句|','|---|---|---|---|']
for a in dry_run['jobs']:
    rows.append('|'+a['role']+'|'+a['voiceId']+'|'+a['lineId']+'|'+a['text']+'|')
display(Markdown('\n'.join(rows)))
print('總共6句：Rosie3句、Princess3句。已生成0句；沒有載入TTS模型。')
print('Rosie ref_text =',repr(library_manifest['characters']['rosie']['reference']['ref_text']))
print('請先回傳這六句清單確認，再繼續。')

## 4｜生成（待六句清單確認後才手動啟用；現在請保留關閉）

In [ ]:
RUN_GENERATION=False
CONFIRMATION='' # 清單經家長確認後，才改為 CONFIRM_LIBRARY_SIX
if RUN_GENERATION:
    generated=library.generate(LIBRARY_WORK,LIBRARY_OUTPUT,run=True,confirmation=CONFIRMATION)
    print('已完成',len(generated['assets']),'個音檔；全部等待逐句聽感驗收。')
else:
    print('已停止：RUN_GENERATION=False，未生成。Bakery/其他24句/Park均未觸碰。')

## 5｜生成後：逐句試聽與品質驗收（尚無音檔時只顯示提示）

Rosie：比較核准參考音，確認童真、甜美自然、活潑而清晰；不能只靠高音判斷。Princess：溫柔優雅親切，保持af_bella。兩者都要聽有無漏詞、吞字、多餘聲音、失真或奇怪停頓。未試聽不會自動核准。

In [ ]:
from IPython.display import Audio, display
import ipywidgets as widgets
import shutil, html
result_file=LIBRARY_OUTPUT/'generated-manifest.json'
if not result_file.exists():
    print('目前沒有生成結果。請先完成dry-run清單確認；本格不會生成任何音檔。')
else:
    result=library.read(result_file)
    if result.get('scope')!='library-trial6-review':raise ValueError('非本次Library六句輸出')
    print('目前完成',len(result['assets']),' / 6，全部仍待人工驗收。')
    ref=library_manifest['characters']['rosie']['reference']
    print('核准Rosie主參考：',ref['ref_text'])
    display(Audio(filename=str(LIBRARY_WORK/'references'/ref['file'])))
    reviews={}
    for job in library_jobs:
        a=result['assets'].get(job['assetId'])
        if not a:print('尚缺',job['lineId']);continue
        if a['text']!=job['text'] or a['voiceId']!=job['voiceId'] or library.sha(LIBRARY_OUTPUT/a['src'])!=a['sha256']:raise ValueError('音檔或聲線資料不符')
        print(a['role'],a['voiceId'],'\n',a['lineId'],'\n',a['text'])
        display(Audio(filename=str(LIBRARY_OUTPUT/a['src'])))
        choice=widgets.Dropdown(options=[('尚未驗收','pending'),('接受此句','accepted'),('需要重錄','needs-retake')],description='聽感：')
        notes=widgets.Textarea(description='備註：',placeholder='例如：太成熟、漏詞、聲音不一致')
        reviews[a['assetId']]=(choice,notes);display(choice,notes)
    download=widgets.Button(description='下載音檔與驗收紀錄',layout=widgets.Layout(width='220px'))
    status=widgets.Output()
    def export_review(_):
        with status:
            review={'scope':'library-trial6-listener-review','productionApproved':False,'lines':{i:{'decision':c.value,'notes':n.value} for i,(c,n) in reviews.items()}}
            library.save(LIBRARY_OUTPUT/'voice-review.json',review)
            cards=[]
            for a in result['assets'].values():
                cards.append('<section><h2>'+html.escape(a['role']+' / '+a['voiceId'])+'</h2><p>'+html.escape(a['text'])+'</p><small>'+html.escape(a['lineId'])+'</small><p><audio controls preload="none" src="'+html.escape(a['src'],quote=True)+'"></audio></p></section>')
            page='<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Library六句試音</title><style>body{max-width:850px;margin:auto;padding:20px;font:17px/1.6 system-ui;background:#faf5e9;color:#30263a}section{border:1px solid #aa956e;border-radius:14px;padding:16px;margin:14px 0}audio{max-width:100%}small{overflow-wrap:anywhere}</style><h1>Library六句試音｜待人工驗收</h1><p>不改正式遊戲。請同時保留audio資料夾；解壓縮後開啟此HTML即可試聽。</p>'+''.join(cards)+'</html>'
            (LIBRARY_OUTPUT/'review.html').write_text(page,encoding='utf-8')
            print('下載目前完成結果：',len(result['assets']),' / 6。未選的仍是pending，不代表正式核准。')
            files.download(shutil.make_archive('/content/library-trial6-results-'+uuid.uuid4().hex[:8],'zip',LIBRARY_OUTPUT))
    download.on_click(export_review);display(download,status)

## 限制與後續

本Notebook有語法/範圍防護/模型呼叫契約測試，但尚未在Colab GPU實際推論本次六句，不宣稱已生成或品質已通過。若Colab中斷，同一執行階段重跑生成格可略過hash吻合的完成句；新執行階段需重新上傳輸入包，未自動還原舊輸出。已下載結果請保留。

只有六句，沒有其他24句/Park/全劇開關。通過人工試音後再討論完整Library與動畫整合；events欄保留原事件但本試音頁不播放遊戲動畫，visemes空白不宣稱音素對齊。

模型/API：[Qwen](https://github.com/QwenLM/Qwen3-TTS)、[Kokoro](https://github.com/hexgrad/kokoro)；授權沿用輸入包LICENSES.md。